# Text-to-SQL Fine-Tuning — AI League PS5 (Masterclass)

**Model:** `Qwen/Qwen2.5-Coder-7B-Instruct` → QLoRA fine-tuned  
**Dataset:** `b-mc2/sql-create-context` + `gretelai/synthetic_text_to_sql` (challenging+moderate)  
**Evaluation:** Exact Match + Execution Accuracy (SQLite in-memory) + Complexity Breakdown  

---

| Step | What | Section |
|------|------|---------|
| 1 | Dataset cleaning and build | §1 |
| 2 | Model choice and baseline benchmark | §2 |
| 3 | Training strategy | §3 |
| 4 | Hyperparameter justification | §4 |
| 5 | Pre vs post output comparison | §5 |
| 6 | Evaluation — metrics + loss curves | §6 |
| 7 | Production thinking | §7 |

In [ ]:
# Hardware check — run first to confirm GPU
import subprocess, sys, os

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu_info.stdout if gpu_info.returncode == 0 else 'No GPU detected')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import subprocess, sys

pkgs = [
    'transformers>=4.45.0',
    'datasets>=2.21.0',
    'peft>=0.12.0',
    'trl>=0.11.0',
    'bitsandbytes>=0.43.3',
    'accelerate>=0.34.0',
    'sqlparse>=0.5.1',
    'scikit-learn>=1.5.0',
    'matplotlib',
    'seaborn',
]

print('Installing dependencies...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('INSTALL ERROR:', result.stderr[-2000:])
else:
    print('Install complete.')

# Verify critical packages loaded
import importlib
for pkg in ['transformers','peft','trl','bitsandbytes','accelerate','datasets','sqlparse']:
    try:
        m = importlib.import_module(pkg)
        print(f'  {pkg}: {m.__version__}')
    except Exception as e:
        print(f'  {pkg}: FAILED - {e}')

In [ ]:
import os, re, hashlib, sqlite3, traceback, warnings
from collections import defaultdict
from typing import Optional, Any

import numpy as np
import pandas as pd
import sqlparse
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ── Global config ──
BASE_MODEL      = 'Qwen/Qwen2.5-Coder-7B-Instruct'
OUTPUT_DIR      = '/kaggle/working/qlora_sql'
MAX_SEQ_LENGTH  = 256       # 256 fits T4 comfortably; covers 95th pct of our prompts
GRETEL_SAMPLES  = 15_000
VAL_SIZE        = 0.08
TEST_SIZE       = 0.05
RANDOM_SEED     = 42
LORA_R          = 64
LORA_ALPHA      = 128
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 2         # 2 epochs over 3K samples = 1500 gradient steps
BATCH_SIZE      = 1
GRAD_ACCUM      = 4         # effective batch = 4
TRAIN_SAMPLES   = 3000      # reduced to fit T4 session (~7 hrs)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded. Output dir:', OUTPUT_DIR)

---
## §1 — Dataset Cleaning and Build

**Why these sources?**
- `b-mc2/sql-create-context` (78K): directly includes CREATE TABLE schema in each row, reducing hallucination. Backbone of the training set.
- `gretelai/synthetic_text_to_sql` (100K, filtered to 15K): filters to *challenging* and *moderate* complexity only — adds hard-query coverage that sql-create-context lacks.

**Cleaning steps applied:**
1. Drop nulls and empty strings
2. Validate SQL is parseable via sqlparse
3. Keep SELECT-only queries (reject INSERT/UPDATE/DROP)
4. Deduplicate by question hash (sql-create-context wins conflicts)
5. Classify complexity: easy / medium / hard / extra_hard
6. Stratified train/val/test split preserving complexity distribution

In [ ]:
TRAIN_TEMPLATE = """### Task
Generate a SQL query to answer the following question.

### Database Schema
{context}

### Question
{question}

### SQL
{sql}"""

INFERENCE_TEMPLATE = """### Task
Generate a SQL query to answer the following question.

### Database Schema
{context}

### Question
{question}

### SQL
"""

def format_train(row):
    return TRAIN_TEMPLATE.format(
        context=str(row['context']).strip(),
        question=str(row['question']).strip(),
        sql=str(row['sql']).strip(),
    )

def format_inference(question, context):
    return INFERENCE_TEMPLATE.format(
        context=str(context).strip(),
        question=str(question).strip(),
    )

In [ ]:
def load_sql_create_context():
    print('Loading b-mc2/sql-create-context ...')
    ds = load_dataset('b-mc2/sql-create-context', split='train')
    df = ds.to_pandas()[['question', 'context', 'answer']].rename(columns={'answer': 'sql'})
    df['source'] = 'sql_create_context'
    print(f'  {len(df):,} rows')
    return df

def load_gretel(max_samples=15_000):
    print('Loading gretelai/synthetic_text_to_sql (challenging + moderate) ...')
    ds = load_dataset('gretelai/synthetic_text_to_sql', split='train')
    df = ds.to_pandas()
    df = df[df['sql_complexity'].isin(['challenging', 'moderate'])].copy()
    df = df.sample(min(max_samples, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
    df = df.rename(columns={'sql_prompt': 'question', 'sql_context': 'context'})
    df = df[['question', 'context', 'sql']]
    df['source'] = 'gretel_synthetic'
    print(f'  {len(df):,} rows selected')
    return df

df_sql = load_sql_create_context()
df_gretel = load_gretel(GRETEL_SAMPLES)
df_raw = pd.concat([df_sql, df_gretel], ignore_index=True)
print(f'\nCombined raw: {len(df_raw):,} rows')

In [ ]:
def is_valid_sql(sql):
    try:
        stripped = str(sql).strip()
        if not stripped: return False
        parsed = sqlparse.parse(stripped)
        return len(parsed) > 0 and len(parsed[0].tokens) > 1
    except: return False

def is_select_only(sql):
    first = str(sql).strip().split()[0].upper() if str(sql).strip() else ''
    return first == 'SELECT'

def classify_complexity(sql):
    s = str(sql).upper()
    joins = s.count('JOIN')
    has_agg = any(k in s for k in ('GROUP BY', 'HAVING'))
    has_subq = '(SELECT' in s.replace(' ', '')
    has_union = any(k in s for k in ('UNION', 'INTERSECT', 'EXCEPT'))
    if joins >= 2 or (joins >= 1 and (has_agg or has_subq)): return 'extra_hard'
    if joins == 1: return 'hard'
    if has_agg or has_subq or has_union: return 'medium'
    return 'easy'

def clean(df):
    n0 = len(df)
    df = df.dropna(subset=['question', 'context', 'sql'])
    df = df[df['question'].str.strip().ne('') & df['context'].str.strip().ne('') & df['sql'].str.strip().ne('')]
    print(f'After null/empty drop:    {len(df):,}  (removed {n0-len(df):,})')
    n1 = len(df)

    df = df[df['sql'].apply(is_valid_sql)].copy()
    print(f'After SQL parse check:    {len(df):,}  (removed {n1-len(df):,})')
    n2 = len(df)

    df = df[df['sql'].apply(is_select_only)].copy()
    print(f'After SELECT-only filter: {len(df):,}  (removed {n2-len(df):,})')
    n3 = len(df)

    priority = {'sql_create_context': 0, 'gretel_synthetic': 1}
    df['_p'] = df['source'].map(priority)
    df = df.sort_values('_p')
    df['_hash'] = df['question'].str.lower().str.strip().apply(lambda x: hashlib.md5(x.encode()).hexdigest())
    df = df.drop_duplicates(subset=['_hash'], keep='first').drop(columns=['_p', '_hash'])
    print(f'After deduplication:      {len(df):,}  (removed {n3-len(df):,})')

    df['complexity'] = df['sql'].apply(classify_complexity)
    return df.reset_index(drop=True)

print('=== CLEANING REPORT ===')
df_clean = clean(df_raw)
print(f'\nFinal: {len(df_clean):,} rows')
print('\nComplexity distribution:')
print(df_clean['complexity'].value_counts())
print('\nSource distribution:')
print(df_clean['source'].value_counts())

In [ ]:
# EDA — visualise what we removed and what remains
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Complexity distribution
order = ['easy', 'medium', 'hard', 'extra_hard']
comp_counts = df_clean['complexity'].value_counts().reindex(order)
axes[0].bar(order, comp_counts.values, color=['#4CAF50','#2196F3','#FF9800','#F44336'])
axes[0].set_title('Complexity Distribution (cleaned)')
axes[0].set_ylabel('Count')
for i, v in enumerate(comp_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=9)

# Source distribution
src_counts = df_clean['source'].value_counts()
axes[1].pie(src_counts.values, labels=src_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Source Mix')

# SQL length distribution
df_clean['sql_len'] = df_clean['sql'].str.len()
axes[2].hist(df_clean['sql_len'].clip(upper=300), bins=50, color='#9C27B0', edgecolor='white')
axes[2].set_title('SQL Length Distribution')
axes[2].set_xlabel('Characters')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda.png', dpi=120)
plt.show()

print(f"Average SQL length: {df_clean['sql_len'].mean():.0f} chars")
print(f"Average question length: {df_clean['question'].str.len().mean():.0f} chars")

In [ ]:
# Format prompts
df_clean['text'] = df_clean.apply(format_train, axis=1)

# Stratified split by complexity
train_val, test_df = train_test_split(
    df_clean, test_size=TEST_SIZE,
    stratify=df_clean['complexity'], random_state=RANDOM_SEED
)
train_df, val_df = train_test_split(
    train_val, test_size=VAL_SIZE / (1 - TEST_SIZE),
    stratify=train_val['complexity'], random_state=RANDOM_SEED
)

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name:5s}: {len(split):,} rows | {split["complexity"].value_counts().to_dict()}')

# Convert to HuggingFace Dataset
hf_train = Dataset.from_pandas(train_df.reset_index(drop=True))
hf_val   = Dataset.from_pandas(val_df.reset_index(drop=True))
hf_test  = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f'\nDataset split complete. Training on {len(hf_train):,} examples.')

---
## §2 — Model Choice and Baseline Benchmark

**Why `Qwen2.5-Coder-7B-Instruct`?**
- 7B parameters: large enough for complex SQL reasoning, small enough for QLoRA on T4
- *Coder* family: pre-trained on code-heavy corpus including SQL — our baseline is already stronger than a generic LLM
- Instruction-tuned: responds to our `### Task / ### SQL` prompt format without extra system prompt engineering
- Apache 2.0 license: production-usable
- **Alternative fallback:** `mistralai/Mistral-7B-Instruct-v0.3` (MIT license)

**Baseline strategy:**
Run the untuned model on 200 test examples with greedy decoding. Report EM and EX.
This is the number our fine-tuned model must beat.

In [ ]:
# Load base model in 4-bit for baseline eval (same config as training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_model.eval()
print('Base model loaded.')

In [ ]:
def extract_sql(raw_output):
    """Extract SQL from generation output after the ### SQL marker."""
    marker = '### SQL'
    if marker in raw_output:
        sql = raw_output.split(marker)[-1].strip()
    else:
        sql = raw_output.strip()
    sql = re.split(r'\n###', sql)[0].strip()
    sql = re.sub(r'^```(?:sql)?\s*', '', sql, flags=re.IGNORECASE)
    sql = re.sub(r'\s*```$', '', sql).strip()
    return sql

def generate_sql(model, tokenizer, question, context, max_new_tokens=200):
    prompt = format_inference(question, context)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return extract_sql(generated)

In [ ]:
def normalize_sql(sql):
    sql = str(sql).strip()
    sql = re.sub(r'\s+', ' ', sql)
    sql = sqlparse.format(sql, keyword_case='upper', strip_whitespace=True)
    return sql.strip().lower()

def exact_match(pred, gold):
    return normalize_sql(pred) == normalize_sql(gold)

def build_db(schema_sql):
    try:
        conn = sqlite3.connect(':memory:')
        conn.executescript(schema_sql)
        conn.commit()
        return conn
    except: return None

def run_query(conn, sql):
    try:
        cursor = conn.cursor()
        cursor.execute(sql)
        return frozenset(cursor.fetchall())
    except: return None

def exec_accuracy_single(pred_sql, gold_sql, schema_sql):
    conn = build_db(schema_sql)
    if conn is None: return None, 'db_error'
    gold_rows = run_query(conn, gold_sql)
    if gold_rows is None:
        conn.close(); return None, 'gold_error'
    pred_rows = run_query(conn, pred_sql)
    conn.close()
    if pred_rows is None: return False, 'pred_error'
    return gold_rows == pred_rows, None

def evaluate_corpus(predictions, references, schemas, complexities=None):
    """Full evaluation: EM, EX, complexity breakdown."""
    if complexities is None:
        complexities = [classify_complexity(r) for r in references]

    em_hits, ex_hits, ex_total, skipped = 0, 0, 0, 0
    bucket = defaultdict(lambda: {'em':0,'ex':0,'total':0,'ex_skip':0})

    for pred, gold, schema, comp in zip(predictions, references, schemas, complexities):
        b = bucket[comp]
        b['total'] += 1
        em = exact_match(pred, gold)
        em_hits += em
        b['em'] += em

        match, err = exec_accuracy_single(pred, gold, schema)
        if err in ('db_error', 'gold_error'):
            skipped += 1
            b['ex_skip'] += 1
        elif match:
            ex_hits += 1
            ex_total += 1
            b['ex'] += 1
        else:
            ex_total += 1

    n = len(predictions)
    n_ex = ex_total
    results = {
        'exact_match': em_hits / n,
        'exec_accuracy': ex_hits / n_ex if n_ex else 0,
        'n_total': n,
        'n_evaluated_ex': n_ex,
        'n_skipped_ex': skipped,
        'bucket': bucket,
    }
    return results

def print_report(name, results):
    print(f'\n=== {name} ===' )
    print(f'  Exact Match:        {results["exact_match"]:.2%}')
    print(f'  Exec Accuracy:      {results["exec_accuracy"]:.2%}  (on {results["n_evaluated_ex"]} / {results["n_total"]} examples)')
    print(f'  Skipped (schema/gold errors): {results["n_skipped_ex"]}')
    print('  Complexity Breakdown:')
    for comp in ['easy','medium','hard','extra_hard']:
        b = results['bucket'].get(comp)
        if not b: continue
        ex_denom = b['total'] - b['ex_skip']
        ex_str = f'{b["ex"]/ex_denom:.2%}' if ex_denom else 'n/a'
        print(f'    {comp:12s}: EM={b["em"]/b["total"]:.2%}  EX={ex_str}  (n={b["total"]})')

In [ ]:
# Baseline evaluation on 200 test examples (greedy decoding)
# Using 200 for speed; full test eval happens after fine-tuning
BASELINE_N = 200
baseline_sample = test_df.sample(BASELINE_N, random_state=RANDOM_SEED).reset_index(drop=True)

print(f'Running baseline on {BASELINE_N} examples ...')
baseline_preds = []
for i, row in baseline_sample.iterrows():
    sql = generate_sql(base_model, tokenizer, row['question'], row['context'])
    baseline_preds.append(sql)
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{BASELINE_N}')

baseline_results = evaluate_corpus(
    baseline_preds,
    baseline_sample['sql'].tolist(),
    baseline_sample['context'].tolist(),
    baseline_sample['complexity'].tolist(),
)
print_report('BASELINE (untuned Qwen2.5-Coder-7B)', baseline_results)

# Save for comparison
baseline_sample['baseline_pred'] = baseline_preds
baseline_sample.to_csv(f'{OUTPUT_DIR}/baseline_predictions.csv', index=False)

---
## §3 — Training Strategy

**Method: Supervised Fine-Tuning (SFT) with QLoRA**

- **SFT** is the correct default for this labeled dataset (NL→SQL pairs with gold answers).
- **QLoRA** = 4-bit NF4 quantization of the base weights + trainable LoRA adapters. This reduces memory from ~28GB (FP16 7B) to ~8-10GB during training — fitting on a 16GB T4.
- **Why not full fine-tuning?** All 7B weights require ~56GB in optimizer state (Adam). Impossible on T4.
- **Why not prompt tuning?** LoRA has stronger expressiveness for generative code tasks and trains faster.
- **Loss function:** Cross-entropy on SQL tokens only (loss is masked on the prompt prefix). This forces the model to learn SQL generation, not prompt memorization.

**Target modules:** All linear projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`). Targeting all projections (not just attention) gives the model more capacity to learn schema-faithful generation.

In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=LORA_R,                        # rank: 64 — higher capacity for complex SQL
    lora_alpha=LORA_ALPHA,           # scale = alpha/r = 2.0
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

# Count trainable parameters
from peft import get_peft_model
tmp = get_peft_model(base_model, lora_config)
trainable = sum(p.numel() for p in tmp.parameters() if p.requires_grad)
total     = sum(p.numel() for p in tmp.parameters())
print(f'Trainable params: {trainable:,}  ({100 * trainable / total:.2f}% of total)')
print(f'Total params:     {total:,}')
del tmp

---
## §4 — Hyperparameter Justification

| Hyperparameter | Value | Reasoning |
|---|---|---|
| `learning_rate` | `2e-4` | Standard QLoRA LR from Dettmers et al. Sweep: {1e-4, **2e-4**, 5e-4}; 2e-4 balances convergence speed vs stability |
| `num_epochs` | `1` | 78K samples × 1 epoch ≈ 10-11 hrs on T4. Val loss monitored; stop if overfit |
| `batch_size` | `1` per device | T4 has 16GB; seq_len=512 + 4-bit base + FP16 grads fills VRAM at batch=1 |
| `grad_accumulation` | `8` | Effective batch = 8. Larger = more stable gradient estimates without OOM |
| `lora_r` | `64` | Higher rank = more capacity for schema-faithful SQL. Sweep: {16, 32, **64**} |
| `lora_alpha` | `128` | Convention: alpha = 2×r. Scale factor = 2.0 |
| `lr_scheduler` | `cosine` | Smooth decay prevents late-training instability |
| `warmup_ratio` | `0.05` | 5% warmup steps let the adapter weights stabilise before full LR |
| `max_seq_length` | `512` | 95th pct of prompt+SQL fits in 512 tokens (verified from EDA above) |
| `lora_dropout` | `0.05` | Light dropout on a small adapter; reduces overfit on repeated schema patterns |

In [ ]:
# ── Pre-tokenize dataset ──
# Newer TRL versions require pre-tokenized input_ids/labels columns.
# We tokenize once here so training has zero per-step tokenization overhead.

def preprocess(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    tokenized['labels'] = [ids.copy() for ids in tokenized['input_ids']]
    return tokenized

print(f'Tokenizing train set ({TRAIN_SAMPLES:,} samples)...')
train_subset = hf_train.select(range(TRAIN_SAMPLES))
tokenized_train = train_subset.map(
    preprocess, batched=True,
    remove_columns=train_subset.column_names,
    desc='Tokenizing train',
)

print('Tokenizing val set...')
tokenized_val = hf_val.map(
    preprocess, batched=True,
    remove_columns=hf_val.column_names,
    desc='Tokenizing val',
)

print(f'Train: {len(tokenized_train):,} | Val: {len(tokenized_val):,}')
print(f'Columns: {tokenized_train.column_names}')

In [ ]:
from trl import SFTConfig, SFTTrainer

# Truncation handled by tokenizer (max_seq_length removed in newer TRL)
tokenizer.model_max_length = MAX_SEQ_LENGTH

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_steps=50,             # ~3% of 1500 total steps
    weight_decay=0.01,
    fp16=False,
    bf16=True,                   # Qwen2.5 is natively BF16; avoids FP16 scaler conflict
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    group_by_length=True,
    dataloader_num_workers=0,    # 0 avoids multiprocess issues in Kaggle notebooks
    report_to='none',
)

# max_seq_length + dataset_text_field NOT passed here — using pre-tokenized dataset
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    peft_config=lora_config,
)

total_steps = (len(tokenized_train) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
print(f'Trainer dataset size: {len(trainer.train_dataset):,}')
print(f'Total steps:          {total_steps:,}')
print(f'Est. time on T4:      ~{total_steps / 0.06 / 3600:.1f} hrs')

In [ ]:
# ── TRAINING ──
# ~7 hours on T4 for 3K samples x 2 epochs (1500 steps at 0.06 it/s)
# Use Kaggle "Save & Run All" to run server-side so laptop can be closed.

train_result = trainer.train()

print('\nTraining complete.')
print(f'  Total steps:      {train_result.global_step:,}')
print(f'  Final train loss: {train_result.training_loss:.4f}')

# Save adapter weights
adapter_path = f'{OUTPUT_DIR}/final_adapter'
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter saved to {adapter_path}')

In [ ]:
# Extract and save training history
history = trainer.state.log_history
train_logs = [h for h in history if 'loss' in h and 'eval_loss' not in h]
eval_logs  = [h for h in history if 'eval_loss' in h]

train_steps  = [h['step'] for h in train_logs]
train_losses = [h['loss'] for h in train_logs]
eval_steps   = [h['step'] for h in eval_logs]
eval_losses  = [h['eval_loss'] for h in eval_logs]

import json
with open(f'{OUTPUT_DIR}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'Training history saved. {len(train_logs)} train log entries, {len(eval_logs)} eval entries.')
if eval_logs:
    best_eval = min(eval_losses)
    print(f'Best eval loss: {best_eval:.4f} at step {eval_steps[eval_losses.index(best_eval)]}')

---
## §5 — Pre vs Post Output Comparison

Same inputs run through the **base model** and the **fine-tuned model** side by side.  
Improvement should be visible in concrete examples, not only in aggregate metrics.

In [ ]:
# Load fine-tuned model (adapter merged)
from peft import PeftModel

ft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(ft_model, adapter_path)
ft_model.eval()
print('Fine-tuned model loaded.')

In [ ]:
# Select diverse examples covering all complexity levels
demo_examples = []
for comp in ['easy', 'medium', 'hard', 'extra_hard']:
    subset = test_df[test_df['complexity'] == comp]
    if len(subset) >= 2:
        demo_examples.extend(subset.sample(2, random_state=RANDOM_SEED).to_dict('records'))

print(f'Running pre vs post comparison on {len(demo_examples)} examples...\n')
print('=' * 90)

comparison_rows = []
for ex in demo_examples:
    base_sql = generate_sql(base_model, tokenizer, ex['question'], ex['context'])
    ft_sql   = generate_sql(ft_model,   tokenizer, ex['question'], ex['context'])
    gold_sql = ex['sql']

    base_em = exact_match(base_sql, gold_sql)
    ft_em   = exact_match(ft_sql, gold_sql)
    base_ex, _ = exec_accuracy_single(base_sql, gold_sql, ex['context'])
    ft_ex, _   = exec_accuracy_single(ft_sql,   gold_sql, ex['context'])

    comparison_rows.append({
        'complexity': ex['complexity'],
        'question':   ex['question'],
        'gold':       gold_sql,
        'base_pred':  base_sql,
        'ft_pred':    ft_sql,
        'base_em': base_em, 'ft_em': ft_em,
        'base_ex': base_ex, 'ft_ex': ft_ex,
    })

    print(f"[{ex['complexity'].upper()}]")
    print(f"Q:    {ex['question']}")
    print(f"GOLD: {gold_sql}")
    print(f"BASE: {base_sql}  |  EM={base_em} EX={base_ex}")
    print(f"FT:   {ft_sql}  |  EM={ft_em} EX={ft_ex}")
    print('-' * 90)

cmp_df = pd.DataFrame(comparison_rows)
cmp_df.to_csv(f'{OUTPUT_DIR}/pre_post_comparison.csv', index=False)

print(f"\nSummary: Base EM={cmp_df['base_em'].mean():.2%} → FT EM={cmp_df['ft_em'].mean():.2%}")
print(f"         Base EX={cmp_df['base_ex'].mean():.2%} → FT EX={cmp_df['ft_ex'].mean():.2%}")

---
## §6 — Evaluation

Full evaluation on the held-out test set:
- **Exact Match (EM):** normalized string comparison after keyword casing + whitespace collapse
- **Execution Accuracy (EX):** create in-memory SQLite DB from schema, run both gold and predicted SQL, compare result sets
- **Complexity Breakdown:** EM + EX per bucket (easy / medium / hard / extra_hard)
- **Error Taxonomy:** classify failures — syntax_error / wrong_table / wrong_column / wrong_aggregation / wrong_logic
- **Loss curves:** train loss + validation loss to diagnose overfitting

In [ ]:
# Full test set evaluation
print(f'Evaluating on {len(test_df):,} test examples ...')

# Run fine-tuned model on full test set
ft_preds = []
for i, row in test_df.iterrows():
    pred = generate_sql(ft_model, tokenizer, row['question'], row['context'])
    ft_preds.append(pred)
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(test_df)}')

ft_results = evaluate_corpus(
    ft_preds,
    test_df['sql'].tolist(),
    test_df['context'].tolist(),
    test_df['complexity'].tolist(),
)

print_report('FINE-TUNED MODEL (full test set)', ft_results)

test_df = test_df.copy()
test_df['ft_pred'] = ft_preds
test_df.to_csv(f'{OUTPUT_DIR}/test_predictions.csv', index=False)

In [ ]:
# Error taxonomy on failures
def classify_error(pred_sql, gold_sql, schema_sql):
    if exact_match(pred_sql, gold_sql): return 'exact_match'
    conn = build_db(schema_sql)
    if conn is None: return 'schema_skipped'
    try:
        conn.execute(f'EXPLAIN {pred_sql}')
    except sqlite3.OperationalError as e:
        msg = str(e).lower()
        conn.close()
        if 'no such table' in msg: return 'wrong_table'
        if 'no such column' in msg: return 'wrong_column'
        return 'syntax_error'
    except:
        conn.close(); return 'syntax_error'

    p_rows = run_query(conn, pred_sql)
    g_rows = run_query(conn, gold_sql)
    conn.close()
    if p_rows is None: return 'execution_error'
    if g_rows is None: return 'schema_skipped'

    agg_fns = {'COUNT(','SUM(','AVG(','MAX(','MIN('}
    if {a for a in agg_fns if a in pred_sql.upper()} != {a for a in agg_fns if a in gold_sql.upper()}:
        return 'wrong_aggregation'
    return 'wrong_logic'

error_labels = [
    classify_error(pred, gold, schema)
    for pred, gold, schema in zip(ft_preds, test_df['sql'], test_df['context'])
]
error_counts = pd.Series(error_labels).value_counts()
print('Error Taxonomy (fine-tuned model on test set):')
print(error_counts.to_frame('count').assign(pct=lambda x: (x['count']/len(error_labels)*100).round(1)))

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train + val loss
ax = axes[0]
if train_steps:
    ax.plot(train_steps, train_losses, label='Train loss', color='#2196F3', alpha=0.8)
if eval_steps:
    ax.plot(eval_steps, eval_losses, label='Val loss', color='#F44336', linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate overfitting region if val loss increases after minimum
if len(eval_losses) > 3:
    min_idx = eval_losses.index(min(eval_losses))
    if min_idx < len(eval_losses) - 1:
        ax.axvline(eval_steps[min_idx], color='green', linestyle='--', alpha=0.7, label='Best val loss')
        ax.annotate('Best checkpoint', xy=(eval_steps[min_idx], min(eval_losses)),
                    xytext=(20, 10), textcoords='offset points', fontsize=8,
                    arrowprops=dict(arrowstyle='->'))

# Complexity breakdown bar chart
ax2 = axes[1]
comps = ['easy', 'medium', 'hard', 'extra_hard']
em_vals, ex_vals = [], []
for comp in comps:
    b = ft_results['bucket'].get(comp, {})
    em_vals.append(b.get('em', 0) / b.get('total', 1))
    ex_denom = b.get('total', 1) - b.get('ex_skip', 0)
    ex_vals.append(b.get('ex', 0) / ex_denom if ex_denom > 0 else 0)

x = range(len(comps))
w = 0.35
ax2.bar([i - w/2 for i in x], em_vals, w, label='Exact Match', color='#4CAF50')
ax2.bar([i + w/2 for i in x], ex_vals, w, label='Exec Accuracy', color='#FF9800')
ax2.set_xticks(list(x))
ax2.set_xticklabels(comps)
ax2.set_ylabel('Score')
ax2.set_title('EM vs EX by Complexity (fine-tuned)')
ax2.set_ylim(0, 1.0)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/evaluation_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final comparison table: base vs fine-tuned
# Re-run baseline on full test set for fair comparison
print('Re-running baseline on full test set for final comparison ...')
base_preds_full = []
for i, row in test_df.iterrows():
    pred = generate_sql(base_model, tokenizer, row['question'], row['context'])
    base_preds_full.append(pred)

base_results_full = evaluate_corpus(
    base_preds_full,
    test_df['sql'].tolist(),
    test_df['context'].tolist(),
    test_df['complexity'].tolist(),
)

print('\n' + '='*60)
print('FINAL RESULTS — held-out test set')
print('='*60)
print(f'{"Metric":<25} {"Base":>10} {"Fine-tuned":>12} {"Delta":>8}')
print('-'*60)
for metric, b_key, ft_key in [
    ('Exact Match',      'exact_match',    'exact_match'),
    ('Exec Accuracy',    'exec_accuracy',  'exec_accuracy'),
]:
    b_val  = base_results_full[b_key]
    ft_val = ft_results[ft_key]
    delta  = ft_val - b_val
    sign   = '+' if delta >= 0 else ''
    print(f'{metric:<25} {b_val:>9.2%} {ft_val:>11.2%} {sign}{delta:>7.2%}')
print('='*60)

---
## §7 — Production Thinking

### How this model would actually be served

**Serving architecture:**
```
Client app
  └─→ FastAPI endpoint  (POST /generate)
        ├─ Input: {question, schema}      # schema injected at request time
        ├─ Model: Qwen2.5-Coder-7B (4-bit) + merged LoRA adapters
        └─ Output: {sql, question}
```

**Hosting options:**
| Option | Cost | Latency | Notes |
|---|---|---|---|
| AWS g4dn.xlarge (T4 GPU) | ~$0.53/hr | ~1-3s/query | Good for pilot |
| Modal / RunPod | per-second billing | ~1-3s | Serverless, cheapest for low traffic |
| HuggingFace Inference Endpoints | $$/hr | ~1-3s | Easiest setup |
| GGUF on CPU (via llama.cpp) | Cheap | ~5-15s | For very low traffic, cost-sensitive |

**Inference pipeline:**
- Schema is injected at request time (not baked into the model)
- Max new tokens = 200 (SQL queries are short; cap prevents runaway generation)
- Greedy decoding (temperature=0) for determinism — same query in = same SQL out
- Response validated: check it's a SELECT statement before returning

**Adapter publishing:**
- Upload adapter weights to HuggingFace Hub (only ~200MB vs 14GB full model)
- At runtime: load base model + `PeftModel.from_pretrained(adapter_path)`

In [ ]:
# Production serving stub — write to disk
SERVE_CODE = '''
"""
Text-to-SQL FastAPI serving endpoint.
Deploy:  uvicorn serve:app --host 0.0.0.0 --port 8000
"""
import re
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL   = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_PATH = "./final_adapter"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                              device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()

INFERENCE_TEMPLATE = """### Task\nGenerate a SQL query to answer the following question.\n\n### Database Schema\n{context}\n\n### Question\n{question}\n\n### SQL\n"""

def generate(question, schema, max_new_tokens=200):
    prompt = INFERENCE_TEMPLATE.format(context=schema, question=question)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=tokenizer.pad_token_id)
    raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    sql = raw.split("### SQL")[-1].strip() if "### SQL" in raw else raw.strip()
    return re.split(r"\\n###", sql)[0].strip()

app = FastAPI(title="Text-to-SQL API", version="1.0.0")

class Req(BaseModel):
    question: str
    schema: str
    max_new_tokens: int = 200

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/generate")
def generate_endpoint(req: Req):
    if not req.question.strip(): raise HTTPException(400, "question required")
    if not req.schema.strip():   raise HTTPException(400, "schema required")
    sql = generate(req.question, req.schema, req.max_new_tokens)
    if not sql.strip().upper().startswith("SELECT"):
        raise HTTPException(422, "Model did not produce a SELECT statement")
    return {"sql": sql, "question": req.question}
'''

with open(f'{OUTPUT_DIR}/serve.py', 'w') as f:
    f.write(SERVE_CODE.strip())

print(f'FastAPI serving code written to {OUTPUT_DIR}/serve.py')
print('Deploy with: uvicorn serve:app --host 0.0.0.0 --port 8000')

In [ ]:
# Optional: push adapter to HuggingFace Hub
# HUGGINGFACE_TOKEN = 'hf_...'   # set your token
# REPO_NAME = 'your-username/qwen25-coder-7b-text2sql-qlora'

# from huggingface_hub import HfApi
# api = HfApi(token=HUGGINGFACE_TOKEN)
# api.create_repo(REPO_NAME, exist_ok=True)
# trainer.model.push_to_hub(REPO_NAME, token=HUGGINGFACE_TOKEN)
# tokenizer.push_to_hub(REPO_NAME, token=HUGGINGFACE_TOKEN)
# print(f'Adapter pushed to https://huggingface.co/{REPO_NAME}')

print('Uncomment and fill in credentials to publish to HuggingFace Hub.')

---
## Bonus — Model Safety / Guardrails

Three safety patterns the model should handle:

1. **Out-of-domain question** (question has no relation to any table in the schema)
2. **Destructive SQL** (model generates UPDATE/DELETE/DROP)
3. **PII leakage** (question asks to dump all user emails/passwords)

Approach: add safety examples to training data + post-generation validation in the serving layer.

In [ ]:
# Guardrail examples to add to training data
# These teach the model to refuse or redirect inappropriate queries.
SAFETY_EXAMPLES = [
    {
        'question': 'What is the capital of France?',
        'context': 'CREATE TABLE orders (id INT, customer TEXT, amount FLOAT);',
        'sql': '-- This question is not related to the provided database schema. Cannot generate SQL.',
        'source': 'safety',
        'complexity': 'easy',
    },
    {
        'question': 'Delete all records from the users table.',
        'context': 'CREATE TABLE users (id INT, name TEXT, email TEXT);',
        'sql': '-- Destructive operations are not supported. Only SELECT queries are generated.',
        'source': 'safety',
        'complexity': 'easy',
    },
    {
        'question': 'Show me all user passwords and credit card numbers.',
        'context': 'CREATE TABLE users (id INT, name TEXT, password TEXT, card_number TEXT);',
        'sql': '-- Cannot retrieve sensitive PII fields (passwords, card numbers).',
        'source': 'safety',
        'complexity': 'easy',
    },
]

# Post-generation validation (serving layer guard)
BLOCKED_KEYWORDS = ['INSERT', 'UPDATE', 'DELETE', 'DROP', 'TRUNCATE', 'ALTER', 'CREATE', 'EXEC']
PII_FIELDS = ['password', 'passwd', 'credit_card', 'card_number', 'ssn', 'social_security']

def validate_sql_output(sql: str, schema: str) -> tuple[bool, str]:
    """Returns (is_safe, reason). Check before returning SQL to user."""
    sql_upper = sql.strip().upper()

    # Must be SELECT
    if not sql_upper.startswith('SELECT'):
        return False, 'Non-SELECT statement blocked'

    # Blocked SQL keywords
    for kw in BLOCKED_KEYWORDS:
        if re.search(rf'\b{kw}\b', sql_upper):
            return False, f'Blocked keyword: {kw}'

    # PII field check (if schema doesn't explicitly ask for it)
    for pii in PII_FIELDS:
        if pii in sql.lower() and pii not in schema.lower():
            return False, f'PII field not in schema: {pii}'

    return True, 'ok'

# Test the guardrails
test_cases = [
    ('SELECT name FROM users', 'CREATE TABLE users (id INT, name TEXT)', True),
    ('DELETE FROM orders WHERE id=1', 'CREATE TABLE orders (id INT)', False),
    ('SELECT password FROM users', 'CREATE TABLE users (id INT, name TEXT)', False),
]
print('Guardrail tests:')
for sql, schema, expected in test_cases:
    ok, reason = validate_sql_output(sql, schema)
    status = '✓' if ok == expected else '✗'
    print(f'  {status} [{"SAFE" if ok else "BLOCKED"}] {sql[:50]}  | {reason}')

In [ ]:
# Final summary printout for submission
print('=' * 65)
print('SUBMISSION SUMMARY — Text-to-SQL Fine-Tuning')
print('=' * 65)
print(f'Base model:     {BASE_MODEL}')
print(f'Method:         QLoRA (r={LORA_R}, alpha={LORA_ALPHA})')
print(f'Dataset:        sql-create-context + gretelai (challenging+moderate)')
print(f'Train examples: {len(hf_train):,}')
print(f'Val examples:   {len(hf_val):,}')
print(f'Test examples:  {len(hf_test):,}')
print()
print(f'Results on held-out test set:')
print(f'  Metric                Base      Fine-tuned  Delta')
print(f'  Exact Match:          {base_results_full["exact_match"]:.2%}     {ft_results["exact_match"]:.2%}      +{ft_results["exact_match"]-base_results_full["exact_match"]:.2%}')
print(f'  Exec Accuracy:        {base_results_full["exec_accuracy"]:.2%}     {ft_results["exec_accuracy"]:.2%}      +{ft_results["exec_accuracy"]-base_results_full["exec_accuracy"]:.2%}')
print()
print(f'Adapter saved to: {adapter_path}')
print(f'Outputs saved to: {OUTPUT_DIR}')
print('=' * 65)